# DIY Copilot
## Lesson: RAG Optimization

This lesson is intended to show you how to optimize the retrieval of a coding assistant. Developers ask short, informal questions ("can I log the meeting title?"), while the knowledge base consists of formal policies and guidelines ("SEC-04 – What must never be logged ... personal data beyond the user id"). Bridging this gap is the main challenge of retrieval.

During this lesson you will learn how to ...

- use **embedding task types** to tell the embedding model what the embeddings are used for
- add **contextual chunk headers** so that every chunk knows which document it comes from
- use **query rewriting** to reshape the developer's question into a retrieval-friendly query
- use **HyDE** to generate a hypothetical wiki section and embed it instead of the raw question
- (optional) use a cross-encoder **reranker** to reorder the retrieved chunks

### Set up the environment
Import the necessary libraries, set constants, and define helper functions.

In [ ]:
# >>> SETUP <<<
# In Google Colab, this cell
#   1. clones the workshop repository,
#   2. installs its dependencies into the Python environment of this notebook (with uv),
#   3. makes the shared code (lib) importable.
# Locally (after `uv sync` in the repository), only step 3 is needed.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/openknowledge/diy-copilot-bootcamp"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/diy-copilot")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, str(REPO_DIR / "setup" / "install_dependencies.py")], check=True)
else:
    # The repository root is the first parent directory containing the shared code
    REPO_DIR = next(p for p in Path.cwd().resolve().parents if (p / "lib").is_dir() and (p / "pyproject.toml").exists())

# The repository is not an installed package, so we add it to the module search path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Workshop repository: {REPO_DIR}")

In [2]:
from chromadb import EphemeralClient

from lib.completion import generate_completion
from lib.constants import EVALUATION_DATA_PATH, KNOWLEDGE_BASE_PATH
from lib.environment import load_environment
from lib.evaluation import FetchedChunk, do_top_k_fetching, load_testdata_from_csv, run_evaluation
from lib.rag import do_embed, do_ingestion, list_knowledge_base_files

In [3]:
# Models
default_models = load_environment()
generation_model = default_models["generation_model"]
# Note: exercise 01 uses task types, which are a feature of the Gemini embedding models.
embedding_model = default_models["embedding_model"]
print(f"Generation model: {generation_model}")
print(f"Embedding model:  {embedding_model}")

# Retrieval parameters
DEFAULT_K = 3
DEFAULT_CHUNK_SIZE = 1000
DEFAULT_CHUNK_OVERLAP = 100

# The Gemini free tier allows about 15 requests per minute per model. Evaluations that call
# the LLM for every question (query rewriting, HyDE) pause between the questions. If the limit
# is reached anyway, the helpers in lib wait automatically as long as the API asks for.
SLEEP_SECONDS_BETWEEN_LLM_CALLS = 4

Generation model: gemini/gemini-3.1-flash-lite
Embedding model:  gemini/gemini-embedding-001


In [ ]:
# This will be the chromadb client we use for the knowledge base.
chromadb_client = EphemeralClient()

# All documents of the knowledge base (domain, engineering, security, hr, it - incl. archived versions)
KNOWLEDGEBASE_CONTENT = list_knowledge_base_files(KNOWLEDGE_BASE_PATH)

In [5]:
def print_evaluation(name: str, evaluation_result) -> None:
    """Prints the evaluation result in one line."""
    print(
        f"{name:<32} Hit Rate {evaluation_result.mean_hit_rate:.2f} | "
        f"MRR {evaluation_result.mean_reciprocal_rank:.2f} | "
        f"Precision {evaluation_result.mean_precision:.2f}"
    )

### Evaluate the basic RAG to define the baseline
The ingestion, fetching and evaluation functions from the last lesson are now available in `lib.rag` and `lib.evaluation`.

In this lesson we only use the **hard** questions of the evaluation dataset. They are phrased like real developer questions: informal, as a scenario, without the vocabulary of the documents, or easy to confuse with a similar document (Berlin vs. Munich, meeting rooms vs. desks, current vs. outdated guidelines). On the easy questions, which use the words of the documents, the baseline is already so good that optimizations hardly make a difference.

In [6]:
# We optimize only the retrieval, therefore we do not augment or generate a response
def do_rag(user_input: str, top_k: int) -> list[FetchedChunk]:
    """Runs the retrieval pipeline with a given user input and returns the context.

    Parameters
    ----------
    user_input : str
        The user input to be used for the RAG pipeline
    top_k : int
        The top_k closest chunks will be retrieved
    Returns
    -------
    list[FetchedChunk]
        The fetched chunks
    """
    user_input_embedding = do_embed(user_input, embedding_model)
    return do_top_k_fetching(user_input_embedding=user_input_embedding, top_k=top_k, chromadb_client=chromadb_client)

In [7]:
# Prepare the knowledge base
do_ingestion(
    file_names=KNOWLEDGEBASE_CONTENT,
    data_path=KNOWLEDGE_BASE_PATH,
    embedding_model=embedding_model,
    chromadb_client=chromadb_client,
    chunk_size=DEFAULT_CHUNK_SIZE,
    chunk_overlap=DEFAULT_CHUNK_OVERLAP,
)

In [ ]:
# Load the test data and keep only the hard questions
testdata_items = [
    item
    for item in load_testdata_from_csv(path=EVALUATION_DATA_PATH / "evaluation_dataset.csv")
    if item.difficulty == "hard"
]
print(f"{len(testdata_items)} hard questions")

In [ ]:
# Evaluate the retrieval without optimizations
baseline_result = run_evaluation(top_k=DEFAULT_K, testdata_items=testdata_items, rag_function=do_rag)
print_evaluation("Baseline", baseline_result)

In [10]:
# Which questions fail? Use verbose=True to see the expected and the retrieved origins.
_ = run_evaluation(top_k=DEFAULT_K, testdata_items=testdata_items, rag_function=do_rag, verbose=True)

HIT  What is the maximum duration of a single meeting room booking in Berlin?
     expected domain/room-booking-policy.md:28-33, got ['domain/room-booking-policy.md:28-50', 'domain/archive/room-booking-policy-v2.md:1-27', 'domain/room-booking-policy.md:48-69']
MISS Can I book a meeting room in Berlin from 10:10 to 10:50?
     expected domain/room-booking-policy.md:22-26, got ['domain/room-booking-policy.md:28-50', 'domain/meeting-rooms-berlin.md:1-24', 'domain/room-booking-policy.md:48-69']
HIT  Can meeting rooms in the Berlin office be booked on Saturdays?
     expected domain/room-booking-policy.md:35-39, got ['domain/room-booking-policy.md:28-50', 'domain/room-booking-policy.md:48-69', 'domain/meeting-rooms-berlin.md:1-24']
HIT  How far in advance can I book a meeting room in Berlin?
     expected domain/room-booking-policy.md:41-44, got ['domain/room-booking-policy.md:28-50', 'domain/room-booking-policy.md:48-69', 'domain/meeting-rooms-berlin.md:1-24']
HIT  Why am I not allowed to 

### Exercise 01: Embedding task types
Up to this point, we embedded questions and chunks in exactly the same way. But a short question ("Why can't I ...?") looks very different from the policy section that answers it. Many embedding models therefore support **task types**: the model produces embeddings that are optimized for a specific use case.

The Gemini embedding models support, among others ([docs](https://ai.google.dev/gemini-api/docs/embeddings#task-types)):
- `RETRIEVAL_DOCUMENT`: for the documents (chunks) in the knowledge base
- `RETRIEVAL_QUERY`: for search queries in natural language
- `QUESTION_ANSWERING`: for questions whose answers are contained in the documents

Your task is to ingest the knowledge base with the task type `RETRIEVAL_DOCUMENT` and evaluate the retrieval with both query task types.

In [11]:
# TODO: Ingest the knowledge base with the task type for documents
do_ingestion(
    file_names=KNOWLEDGEBASE_CONTENT,
    data_path=KNOWLEDGE_BASE_PATH,
    embedding_model=embedding_model,
    chromadb_client=chromadb_client,
    task_type="RETRIEVAL_DOCUMENT",
)

In [ ]:
# TODO: Implement a RAG function which embeds the user input with a given query task type
def do_task_type_rag(user_input: str, top_k: int, task_type: str = "RETRIEVAL_QUERY") -> list[FetchedChunk]:
    """Runs the retrieval pipeline and embeds the user input with the given task type."""
    user_input_embedding = do_embed(user_input, embedding_model, task_type=task_type)
    return do_top_k_fetching(user_input_embedding=user_input_embedding, top_k=top_k, chromadb_client=chromadb_client)


# TODO: Evaluate both query task types
print_evaluation("Baseline", baseline_result)
for task_type in ["RETRIEVAL_QUERY", "QUESTION_ANSWERING"]:
    result = run_evaluation(
        top_k=DEFAULT_K,
        testdata_items=testdata_items,
        rag_function=lambda user_input, top_k: do_task_type_rag(user_input, top_k, task_type=task_type),
    )
    print_evaluation(f"Task type {task_type}", result)

### Exercise 02: Contextual chunk headers
Look at a chunk from the middle of a document: it often contains a single section like "## Cancellation", but nothing tells the embedding model *which* cancellation – meeting rooms or desks? Yet the document path carries a lot of meaning: `domain/room-booking-policy.md`, `domain/desk-sharing-policy.md` or `security/data-protection.md`.

A simple but effective trick is to put a header with the document path in front of every chunk before embedding it. `do_ingestion` supports this via the parameter `add_file_header=True`.

Your task is to ingest the knowledge base with file headers and evaluate the retrieval. Compare the result with the baseline.

In [13]:
# TODO: Ingest the knowledge base with contextual chunk headers
do_ingestion(
    file_names=KNOWLEDGEBASE_CONTENT,
    data_path=KNOWLEDGE_BASE_PATH,
    embedding_model=embedding_model,
    chromadb_client=chromadb_client,
    add_file_header=True,
)

[retry] rate limit reached, waiting 24s (attempt 1/7) ...
[retry] rate limit reached, waiting 58s (attempt 2/7) ...


In [14]:
# Let's have a look at a chunk with a header
print(do_rag("Until when can I cancel my desk booking?", top_k=1)[0].chunk)

# File: domain/desk-sharing-policy.md
# Desk Sharing Policy

- Owner: Facility Management
- Applies to: all employees working in the Berlin office
- Version: 1.4

Since 2024 the Berlin office works with shared desks. Desks are booked with
the desk booking app "Deskly" (not with Roomie, which is only used for meeting
rooms).

## Booking a desk

- Desks can be booked at most **14 days in advance**.
- A desk is always booked for a whole day; half-day bookings are not possible.
- Every employee can hold at most **3 desk bookings** in the future at the same time.
- Team areas: each team has a home area where its members should book first.

## Cancellation

Desk bookings can be cancelled until **10:00 on the same day**. Later
cancellations are counted as a no-show.

## Check-in

Desks have to be checked in by scanning the QR code on the desk until
**10:30**. Desks that are not checked in are released at 10:30 and can be
booked by others for the rest of the day.


In [ ]:
# TODO: Evaluate the retrieval with contextual chunk headers
header_result = run_evaluation(top_k=DEFAULT_K, testdata_items=testdata_items, rag_function=do_rag)
print_evaluation("Baseline", baseline_result)
print_evaluation("Contextual chunk headers", header_result)

### Reset the knowledge base
Execute this as soon as you are ready to move on to the next exercise. We go back to the baseline knowledge base, so that the following techniques can be compared with the baseline.

In [16]:
do_ingestion(
    file_names=KNOWLEDGEBASE_CONTENT,
    data_path=KNOWLEDGE_BASE_PATH,
    embedding_model=embedding_model,
    chromadb_client=chromadb_client,
)

[retry] rate limit reached, waiting 52s (attempt 1/7) ...


### Exercise 03: Query Rewriting
A developer's question often doesn't use the words of the knowledge base. "Can I put the API key in my test?" is answered by a section about "secrets", "repositories" and "demo keys". An LLM can rewrite the question into a more retrieval-friendly form – adding the terms a policy or guideline would use, and dropping conversational filler.

Think of this as **HyDE-lite**: same idea of putting an LLM in front of retrieval, but the output stays at the *query* level instead of fabricating a hypothetical answer. It's cheaper (one short generation), and it avoids inventing facts that could anchor retrieval to the wrong neighborhood.

**Your task:** implement `rewrite_query` so that it turns a developer question into a search query for the company knowledge base, then evaluate `do_query_rewrite_rag`.

>**Note:** Query rewriting calls the LLM once per question. Because of the rate limits of the free tier, we evaluate it only on **5 selected questions** and compare it with the baseline on the same 5 questions. The evaluation prints the rewritten query and the result for every question.

In [17]:
# TODO: Implement a function which rewrites a developer question into a more retrieval-friendly query
def rewrite_query(user_input: str) -> str:
    """Rewrites a developer question into a search query for the company knowledge base.

    Parameters
    ----------
    user_input : str
        The user input to be rewritten
    Returns
    -------
    str
        The rewritten query
    """

    # TODO: Use your prompt engineering skills to create a prompt that turns
    # a developer's natural-language question into a search query for the knowledge base.
    #
    # The rewritten query should:
    #   - Preserve the developer's intent and all names given in the question
    #     (rooms, tools, error codes, ...)
    #   - Add the formal terms a policy or guideline would use for the topic
    #     (e.g. "secrets", "personal data", "retention", "approval")
    #   - Remove conversational filler ("could you tell me", "I want to know", etc.)
    #   - Stay one or two lines
    #   - Be returned as plain text only - no quotes, no preamble
    #
    # As an example, you can use the following developer question and the
    # resulting rewritten query:
    #
    # DEVELOPER QUESTION:
    # "Could you tell me why my meeting gets deleted when I come 15 minutes late?"
    #
    # REWRITTEN QUERY:
    # meeting room booking automatically released no check-in no-show check-in time after start

    rewrite_prompt = f"""
You are a search-query rewriter for the internal knowledge base of a software company.
The knowledge base contains business policies (e.g. meeting room booking, desk sharing),
engineering guidelines (Python coding, API design, testing, git and code review) and
security requirements (secrets, logging, data protection, use of AI assistants).

Given a developer question, rewrite it into a single search query that is likely to match
the wording of the policy or guideline that answers the question.

Rules:
- Preserve the developer's intent and all names given in the question (rooms, tools, error codes).
- Add the formal terms a policy or guideline would use for this topic.
- Drop conversational filler ("could you tell me", "I want to know", "please", question marks).
- One or two lines. No quotes. No preamble. No explanation.

Below is an example:

DEVELOPER QUESTION:
Could you tell me why my meeting gets deleted when I come 15 minutes late?

REWRITTEN QUERY:
meeting room booking automatically released no check-in no-show check-in time after start

Now rewrite the following question. Output only the rewritten query.

DEVELOPER QUESTION:
{user_input}

REWRITTEN QUERY:
"""

    return generate_completion(
        user_prompt=rewrite_prompt,
        model_name=generation_model,
        temperature=0.0,
        max_output_tokens=100,
    ).strip()

In [18]:
# The following function uses rewrite_query as a lightweight pre-retrieval step
def do_query_rewrite_rag(user_input: str, top_k: int, verbose: bool = False) -> list[FetchedChunk]:
    """Runs the retrieval pipeline using a single LLM-rewritten query.

    Parameters
    ----------
    user_input : str
        The user input to be used for the RAG pipeline
    top_k : int
        The top_k closest chunks will be retrieved
    Returns
    -------
    list[FetchedChunk]
        The fetched chunks
    """
    # Rewrite the developer's question into a search-friendly query
    rewritten_query = rewrite_query(user_input=user_input)

    # Embed the rewritten query
    rewritten_query_embedding = do_embed(rewritten_query, embedding_model)

    # "R" like "Retrieval": Get the top-k semantically closest chunks to the rewritten query
    fetched_chunks = do_top_k_fetching(
        user_input_embedding=rewritten_query_embedding, top_k=top_k, chromadb_client=chromadb_client
    )

    if verbose:
        print(f"Original:  {user_input}")
        print(f"Rewritten: {rewritten_query}")
        print(f"Fetched chunks: {[chunk.metadata.origin for chunk in fetched_chunks]}")
    return fetched_chunks

In [19]:
# TODO: Test the do_query_rewrite_rag function with a user input
user_input = "Can I put my API key into a test file?"
context = do_query_rewrite_rag(user_input=user_input, top_k=DEFAULT_K, verbose=True)

[retry] temporary error (ServiceUnavailableError), waiting 5s (attempt 1/7) ...
[retry] temporary error (ServiceUnavailableError), waiting 10s (attempt 2/7) ...
Original:  Can I put my API key into a test file?
Rewritten: API key storage security policy hardcoded secrets test files source code repository protection
Fetched chunks: ['security/secrets-and-logging.md:1-23', 'security/ai-assistant-usage-policy.md:40-60', 'security/secrets-and-logging.md:25-48']


In [ ]:
# TODO: Evaluate the new system on 5 selected questions. How does it perform compared to the baseline?
# What might be the reasons for the differences?
# (Only 5 questions, because every question costs one LLM request on the free tier.
#  We selected questions where the baseline struggles.)
rewrite_sample = [testdata_items[i] for i in (6, 10, 20, 22, 26)]
for item in rewrite_sample:
    print("-", item.input)
print()

baseline_sample_result = run_evaluation(top_k=DEFAULT_K, testdata_items=rewrite_sample, rag_function=do_rag)
rewrite_result = run_evaluation(
    top_k=DEFAULT_K,
    testdata_items=rewrite_sample,
    rag_function=lambda user_input, top_k: do_query_rewrite_rag(user_input, top_k, verbose=True),
    sleep_seconds=SLEEP_SECONDS_BETWEEN_LLM_CALLS,
    verbose=True,
)
print()
print_evaluation("Baseline (5 questions)", baseline_sample_result)
print_evaluation("Query rewriting (5 questions)", rewrite_result)

### Exercise 04: Hypothetical Document Embeddings (HyDE)
HyDE is a technique for improving retrieval. Instead of embedding the question, we let an LLM generate a **hypothetical document** that could answer it – in our case: a hypothetical section of the company wiki, written like a policy or guideline. This hypothetical section is then used for the semantic retrieval.

The underlying idea is that, although the generated section will not be factually correct (the LLM doesn't know our rules!), it is often semantically much closer to the real policy than the short question: it uses the same formal language, the same structure and similar terms.

In practice, multiple hypothetical documents are often generated, embedded, and then averaged over each embedding dimension to create a single embedding used for retrieval. This averaging is intended to neutralize incorrect details while enhancing the semantic similarity to the knowledge base.

**Your task is to implement this method.** Afterwards, reflect on the potential drawbacks of this approach and evaluate the performance of the resulting system.

>**Note:** Like query rewriting, HyDE calls the LLM once per question. We therefore evaluate it on the same **5 questions**, so the results of the baseline, query rewriting and HyDE can be compared directly.

In [ ]:
HYDE_GENERATION_MODEL = generation_model

# TODO: Implement a function which creates a hypothetical wiki section for a given user input
def generate_hypothetical_document(user_input: str) -> str:
    """Generates a hypothetical section of the company knowledge base that answers the user input.

    Parameters
    ----------
    user_input : str
        The user input to be used for the generation
    Returns
    -------
    str
        The generated hypothetical section
    """

    # TODO: Use your prompt engineering skills to create a prompt for generating
    # a hypothetical section of the company wiki that answers the question.
    #
    # The generated section must:
    #   - Look as if it was taken from an internal policy or guideline of a software company
    #     (Markdown heading, short paragraphs or bullet points, formal language)
    #   - Contain the information that answers the question (invented details are ok)
    #   - Be short (max. ~15 lines) and contain no explanations outside the section
    #
    # As an example, you can use the following question and the resulting section:
    #
    # QUESTION:
    # "How long do I have to wait before I can book a room again after a meeting with catering?"
    #
    # HYPOTHETICAL SECTION:
    # ### Rooms with catering
    # After every meeting with catering, the cleaning team needs 30 minutes to clean
    # the room. During this time the room is blocked and cannot be booked. The next
    # booking can start 30 minutes after the previous one ends at the earliest.

    hyde_prompt = f"""
You are HYDE, a generator of hypothetical wiki sections. You write sections as they could
appear in the internal knowledge base of a software company. The knowledge base contains
business policies (e.g. meeting room booking, desk sharing), engineering guidelines
(Python coding, API design, testing, git and code review) and security requirements
(secrets, logging, data protection, use of AI assistants).

Given a developer question, write the section of the knowledge base that most likely
contains the answer:
- Markdown heading, short paragraphs or bullet points, formal policy language.
- The section must contain information that answers the question. You may invent plausible details.
- Max. 15 lines. Output only the section, no explanations.

Example:

QUESTION:
How long do I have to wait before I can book a room again after a meeting with catering?

HYPOTHETICAL SECTION:
### Rooms with catering
After every meeting with catering, the cleaning team needs 30 minutes to clean
the room. During this time the room is blocked and cannot be booked. The next
booking can start 30 minutes after the previous one ends at the earliest.

Now generate the section for the following question.

QUESTION:
{user_input}

HYPOTHETICAL SECTION:
"""

    return generate_completion(
        user_prompt=hyde_prompt,
        model_name=HYDE_GENERATION_MODEL,
        temperature=0.7,
        max_output_tokens=400,
    )

In [ ]:
# The following function uses the generate_hypothetical_document function
def do_hyde_rag(user_input: str, top_k: int, n_hyde: int = 1, verbose: bool = False) -> list[FetchedChunk]:
    """Runs the retrieval pipeline using hypothetical document embeddings (HyDE).

    Parameters
    ----------
    user_input : str
        The user input to be used for the RAG pipeline
    top_k : int
        The top_k closest chunks will be retrieved
    n_hyde : int
        The number of hypothetical documents to be created and averaged
    Returns
    -------
    list[FetchedChunk]
        The fetched chunks
    """
    # TODO: Generate hypothetical documents for the user input
    hypothetical_documents = [generate_hypothetical_document(user_input=user_input) for _ in range(n_hyde)]

    # Embed the hypothetical documents
    embeddings = [do_embed(document, embedding_model) for document in hypothetical_documents]

    # Average the embeddings over each dimension
    average_embedding = [sum(values) / n_hyde for values in zip(*embeddings)]

    # "R" like "Retrieval": Get the top-k semantically closest chunks to the hypothetical documents
    fetched_chunks = do_top_k_fetching(user_input_embedding=average_embedding, top_k=top_k, chromadb_client=chromadb_client)

    if verbose:
        print(f"User input: {user_input}")
        print(f"Example of a generated hypothetical document:\n{hypothetical_documents[0]}\n")
        print(f"Fetched chunks: {[chunk.metadata.origin for chunk in fetched_chunks]}")
    return fetched_chunks

In [ ]:
# TODO: Test the do_hyde_rag function with a user input
user_input = "May we build a report of the employees with the most no-shows?"
context = do_hyde_rag(user_input=user_input, top_k=DEFAULT_K, verbose=True)

In [ ]:
# TODO: Evaluate the new system on the same 5 questions. How does it perform compared to the
# baseline and to query rewriting? What might be the reasons for the differences?
# (Only 5 questions, because every question costs one LLM request on the free tier.)
hyde_result = run_evaluation(
    top_k=DEFAULT_K,
    testdata_items=rewrite_sample,
    rag_function=lambda user_input, top_k: do_hyde_rag(user_input, top_k, verbose=True),
    sleep_seconds=SLEEP_SECONDS_BETWEEN_LLM_CALLS,
    verbose=True,
)
print()
print_evaluation("Baseline (5 questions)", baseline_sample_result)
print_evaluation("Query rewriting (5 questions)", rewrite_result)
print_evaluation("HyDE (5 questions)", hyde_result)

### (Optional) Exercise 05: Reranking
In a RAG system, rerankers play a crucial role in improving the quality of retrieved information before it's passed to the language model. While the retriever fetches potentially relevant chunks based on embedding similarity, it often returns noisy or loosely related results. Rerankers evaluate and reorder these chunks using a **cross-encoder**: a model that looks at the question and a chunk *together* and scores how well the chunk answers the question. This is much more accurate than comparing two independent embeddings – but also much slower, so it is only applied to a small set of candidates.

A typical setup: retrieve more candidates than needed (e.g. 3 × top_k), rerank them, and keep only the best top_k.

For this exercise you will load a Hugging Face cross-encoder (no API key needed). `torch` and `transformers` are preinstalled in Colab. If a GPU runtime is available, the reranker uses it automatically.

> **Local setup:** run `uv sync --group reranker` from the project root before executing the cells below.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [ ]:
reranker_model_name = 'BAAI/bge-reranker-v2-m3'
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(reranker_model_name)
reranker_model = AutoModelForSequenceClassification.from_pretrained(reranker_model_name).to(device).eval()
print(f"Reranker running on {device}")


def rerank(user_input: str, fetched_chunks: list[FetchedChunk]) -> list[FetchedChunk]:
    """Reranks fetched chunks using a cross-encoder, most relevant chunk first."""
    pair_inputs = [(user_input, chunk.chunk) for chunk in fetched_chunks]
    with torch.no_grad():
        inputs = tokenizer(pair_inputs, padding=True, truncation=True, return_tensors='pt', max_length=512).to(device)
        scores = reranker_model(**inputs, return_dict=True).logits.view(-1).float().cpu()
    ranked = sorted(zip(scores.tolist(), fetched_chunks), key=lambda item: item[0], reverse=True)
    return [chunk for _, chunk in ranked]

In [ ]:
# TODO: Update the do_reranked_rag function to use the reranking function
def do_reranked_rag(user_input: str, top_k: int, candidate_factor: int = 3, verbose: bool = False) -> list[FetchedChunk]:
    """Retrieves candidate_factor * top_k chunks, reranks them and returns the best top_k.

    Parameters
    ----------
    user_input : str
        The user input to be used for the RAG pipeline
    top_k : int
        The number of chunks to be returned
    candidate_factor : int
        How many more candidates than top_k are retrieved before reranking
    Returns
    -------
    list[FetchedChunk]
        The reranked chunks
    """
    user_input_embedding = do_embed(user_input, embedding_model)

    # "R" like "Retrieval": Get more candidates than we need
    candidates = do_top_k_fetching(
        user_input_embedding=user_input_embedding, top_k=top_k * candidate_factor, chromadb_client=chromadb_client
    )

    # TODO: Rerank the candidates based on the user input and keep the best top_k
    reranked = rerank(user_input=user_input, fetched_chunks=candidates)[:top_k]

    if verbose:
        print(f"User input: {user_input}")
        print(f"Candidates: {[c.metadata.origin for c in candidates]}")
        print(f"Reranked:   {[c.metadata.origin for c in reranked]}")
    return reranked

In [ ]:
_ = do_reranked_rag("Why is there no DELETE endpoint for cancelled bookings?", top_k=DEFAULT_K, verbose=True)

In [ ]:
# TODO: Evaluate the new system. How does it perform compared to the baseline?
rerank_result = run_evaluation(top_k=DEFAULT_K, testdata_items=testdata_items, rag_function=do_reranked_rag)
print_evaluation("Baseline", baseline_result)
print_evaluation("Reranking", rerank_result)

### Learnings
* There is no single best technique – always measure on *your* data. That's why we built the evaluation first.
* Evaluate with **hard** questions: on easy questions (which use the words of the documents) every technique looks good.
* Not every technique helps: in our tests, the task types made hardly any difference, while chunk headers helped with confusable documents (Berlin vs. Munich, rooms vs. desks).
* HyDE helped the most in our tests – the hypothetical section "speaks the language" of the policies. But techniques that call an LLM per query cost latency, money and rate limit – a coding assistant has to answer within seconds.
* Query rewriting can improve the ranking (MRR) but also lead the retrieval astray when the LLM adds the wrong terms.
* Outdated documents (`archive/`) are a real problem: they *sound* right. Metadata (e.g. a status field) and filters help more than better embeddings.
* Production systems often combine several of these techniques – plus classic keyword search, which is unbeatable for exact terms like rule ids (`SEC-04`) or error codes. On day 2 we will give the assistant tools, so it can decide itself *when* and *how* to search the knowledge base – and the code.